In [13]:
%load_ext autoreload
%autoreload 2

from go_ml.model_interp.mut_scan import mut_scan_esm, aa_str
import pickle

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
with open('../model_interp/llps_scan.pkl', 'rb') as f:
    llps_scan = pickle.load(f)

In [12]:
from go_ml.models.warmup_esm_finetune import ESMFinetune
checkpoint_path = "/home/andrew/GO_interp/checkpoints/esm_finetune_fin.ckpt"
model = ESMFinetune.load_from_checkpoint(checkpoint_path, strict=False)
print(f'Loaded model from {checkpoint_path}')

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/andrew/anaconda3/envs/goint/lib/python3.10/site-packages/pytorch_lightning/core/saving.py:191: Found keys that are in the model state dict but not in the checkpoint: ['model.embeddings.position_embeddings.weight']


Loaded model from /home/andrew/GO_interp/checkpoints/esm_finetune_fin.ckpt


In [ ]:
seq = 'LAGVSERTIDPKQNFYMHWCLAGVSERTIDPKQNFYMHWCLAGVSERTIDPKQNFYMHWC'
from go_ml.model_interp.mut_scan import mut_scan_stream, rebatch_stream
import torch
batch_stream = mut_scan_stream(seq, model.tokenizer)
rebatched_stream = rebatch_stream(mut_scan_stream(seq, model.tokenizer), batch_size=8)
batch_stream_mat = torch.concat([batch for batch in batch_stream], dim=0)
rebatched_stream_mat = torch.concat([batch for batch in rebatched_stream], dim=0)

assert batch_stream_mat.shape == rebatched_stream_mat.shape, "Batching did not preserve shape"
assert all((batch_stream_mat == rebatched_stream_mat).flatten()), "Batching did not preserve values"

In [28]:
batch_stream = mut_scan_stream(seq, model.tokenizer)
rebatched_stream = rebatch_stream(mut_scan_stream(seq, model.tokenizer), batch_size=43)
batch_stream_mat = torch.concat([batch for batch in batch_stream], dim=0)
rebatched_stream_mat = torch.concat([batch for batch in rebatched_stream], dim=0)

assert batch_stream_mat.shape == rebatched_stream_mat.shape, "Batching did not preserve shape"
assert all((batch_stream_mat == rebatched_stream_mat).flatten()), "Batching did not preserve values"

In [25]:
for batch in rebatched_stream:
    print(f'Batch shape: {batch.shape}')

Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shape: torch.Size([8, 62])
Batch shap

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(15, 10))
ax = sns.heatmap(eval_mat[:, :, 67].T - np.mean(eval_mat[:, :, 67]), cmap="viridis", cbar=True, cbar_kws={'label': 'Logits'})  
plt.title(f"Associated Function: {go_terms[67]}", fontsize=20)
plt.xlabel("Residue Index", fontsize=20)
plt.ylabel("Amino Acids", fontsize=20)
ax.set_yticklabels(aa_str, fontsize=20)
# ax.set_xticklabels(np.arange(0, eval_mat[:, :, 67].shape[0], 8))
plt.tight_layout()